In [ ]:
from pathlib import Path
import os
import pandas as pd
from IPython.display import display

# Open from the repository root or the training folder.
WORKSPACE = Path.cwd()
if (WORKSPACE / "training/all_benchmarks_flattened_training_keras.ipynb").is_file():
    WORKSPACE = WORKSPACE / "training"
DATA_DIR = Path(os.environ.get("AIMS_DATA_DIR", str(WORKSPACE / "data"))).expanduser()
print(DATA_DIR)
assert DATA_DIR.is_dir(), "Set AIMS_DATA_DIR to the downloaded benchmark folders (see training/README.md)."
pd.set_option("display.max_colwidth", 100)


In [ ]:
# Purpose: Attach lookup metadata to response rows, checking that every row matches exactly once.
def join_lookup(observations, lookup, keys, prefix):
    assert not lookup[keys].isna().any().any(), f"{prefix}: missing lookup keys"
    assert not observations[keys].isna().any().any(), f"{prefix}: missing response keys"
    renamed = lookup.rename(columns={
        column: f"{prefix}_{column}"
        for column in lookup.columns if column not in keys
    })
    joined = observations.merge(
        renamed, on=keys, how="left", validate="many_to_one", indicator="_matched"
    )
    assert len(joined) == len(observations), f"{prefix}: row count changed"
    assert joined["_matched"].eq("both").all(), f"{prefix}: unmatched response"
    return joined.drop(columns="_matched")


In [ ]:
# Purpose: Load one benchmark and combine responses, subjects, items, and benchmark metadata into one table.
def flatten_benchmark(folder):
    tables = {
        name: pd.read_parquet(folder / f"{name}.parquet")
        for name in ["benchmarks", "subjects", "items", "response"]
    }
    responses = tables["response"]
    assert responses["response_id"].notna().all()
    assert responses["response_id"].is_unique

    flattened = join_lookup(responses, tables["subjects"], ["subject_id"], "subject")
    flattened = join_lookup(
        flattened, tables["items"], ["benchmark_id", "item_id"], "item"
    )
    flattened = join_lookup(
        flattened, tables["benchmarks"], ["benchmark_id"], "benchmark"
    )
    flattened.insert(0, "dataset", folder.name)
    assert flattened["response"].equals(responses["response"].reset_index(drop=True))
    return flattened


In [ ]:
benchmark_folders = sorted(
    folder for folder in DATA_DIR.iterdir()
    if folder.is_dir() and (folder / "response.parquet").exists()
)
assert benchmark_folders, "No downloaded benchmark tables found."

flat_tables = {
    folder.name: flatten_benchmark(folder)
    for folder in benchmark_folders
}
flat_all = pd.concat(flat_tables.values(), ignore_index=True, sort=False)
assert len(flat_all) == sum(len(table) for table in flat_tables.values())
assert not flat_all.duplicated(["dataset", "response_id"]).any()

print("Benchmarks:", list(flat_tables))
print("Flattened shape:", flat_all.shape)


In [ ]:
display(flat_all.head())
display(pd.DataFrame({
    "column": flat_all.columns,
    "dtype": flat_all.dtypes.astype(str).to_numpy()
}))


In [ ]:
# Purpose: Summarize dataset sizes, response formats, and missing or nonbinary grades.
def summarize_benchmarks(flattened):
    return flattened.groupby("dataset", sort=True).agg(
        items=("item_id", "nunique"),
        subjects=("subject_id", "nunique"),
        attempts=("response_id", "size"),
        response_type=("benchmark_response_type", "first"),
        granularity=("benchmark_granularity", "first"),
        missing_grades=("response", lambda grades: grades.isna().sum()),
        nonbinary_grades=(
            "response", lambda grades: (grades.notna() & ~grades.isin([0, 1])).sum()
        ),
    )

display(summarize_benchmarks(flat_all))


In [ ]:
# Purpose: Keep graded, item-level binary responses that can train the success-probability model.
def binary_observations(flattened):
    eligible = (
        flattened["benchmark_response_type"].eq("binary")
        & flattened["benchmark_granularity"].eq("item")
        & flattened["response"].notna()
    )
    binary = flattened.loc[eligible].copy()
    assert binary["response"].isin([0, 1]).all()
    return binary

flat_binary = binary_observations(flat_all)
print("All observations:", len(flat_all))
print("Binary-eligible observations:", len(flat_binary))
display(summarize_benchmarks(flat_binary))


In [ ]:
# Purpose: Separate raw subject/item metadata from response labels for inspection before feature encoding.
def approach1_inputs(observations):
    columns = [
        "subject_normalized_name", "subject_provider", "subject_release_date",
        "subject_access_date", "subject_harness", "subject_harness_version",
        "subject_reasoning_effort", "subject_subject_features_extra",
        "item_content", "item_item_features", "interactors",
    ]
    inputs = observations[columns].copy()
    target = observations["response"].copy()
    return inputs, target

X_raw, y = approach1_inputs(flat_binary)
display(X_raw.head())
display(y.head())


In [ ]:
# Setup
import os, sys
if "keras" in sys.modules and sys.modules["keras"].backend.backend() != "torch":
    raise RuntimeError("Restart the kernel before selecting the torch backend.")
os.environ["KERAS_BACKEND"] = "torch"
import hashlib, importlib, importlib.util, json, shutil, subprocess, zipfile
import numpy as np
import torch
import keras
from sklearn.model_selection import train_test_split

if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))
import feature_utils as feature_code
import keras_submission as keras_code
importlib.reload(feature_code)
importlib.reload(keras_code)
from feature_utils import MetadataEncoder, SUBJECT_COLUMNS, make_text_encoder, prepare_chunk_features, prompt_chunks, embed_text

TEXT_ENCODER_DIR = Path(os.environ.get("AIMS_ENCODER_DIR", str(WORKSPACE / "resources/text_encoder"))).expanduser()
KIT_DIR = Path(os.environ.get("AIMS_STARTER_KIT_DIR", str(WORKSPACE / "official-starting-kit"))).expanduser()
TEXT_MAX_LENGTH, TEXT_CHUNK_SIZE, TEXT_BATCH_SIZE = 2048, 512, 32
torch.set_num_threads(4)
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
KERAS_DEVICE = str(DEVICE)
keras.config.set_floatx("float32")
print("Keras:", keras.__version__, "Backend:", keras.backend.backend(), "Device:", DEVICE)


In [ ]:
# Fixed prompt embeddings: reuse the disk cache when prompts and encoder settings match.
from prompt_embedding_cache import cached_prompt_embeddings

unique_items = flat_binary[["item_content"]].copy()
unique_items["item_content"] = unique_items["item_content"].fillna("").astype(str)
unique_items = unique_items.drop_duplicates("item_content")

prompts = unique_items["item_content"].tolist()

# Purpose: Encode each unique prompt with the frozen text encoder when the disk cache needs rebuilding.
def compute_prompt_vectors():
    text_encoder = make_text_encoder(
        TEXT_ENCODER_DIR, TEXT_MAX_LENGTH, device=DEVICE, chunk_size=TEXT_CHUNK_SIZE
    )
    assert text_encoder.device.type == DEVICE.type
    token_lengths = np.array([
        len(text_encoder.tokenizer.encode(
            prompt, add_special_tokens=False, truncation=False, verbose=False
        ))
        for prompt in unique_items["item_content"]
    ])
    special_tokens = text_encoder.tokenizer.num_special_tokens_to_add(pair=False)
    chunk_payload = TEXT_CHUNK_SIZE - special_tokens
    kept_lengths = np.minimum(token_lengths, TEXT_MAX_LENGTH)
    chunk_counts = np.maximum(1, (kept_lengths + chunk_payload - 1) // chunk_payload)
    print("Encoder device:", text_encoder.device)
    print("Prompt token budget:", TEXT_MAX_LENGTH, "Chunk window:", TEXT_CHUNK_SIZE)
    print("Prompts needing multiple chunks:", int((chunk_counts > 1).sum()))
    print("Prompts truncated beyond the total budget:", int((token_lengths > TEXT_MAX_LENGTH).sum()))
    print("Total chunks:", int(chunk_counts.sum()))

    longest_index = int(token_lengths.argmax())
    probe_chunks = list(prompt_chunks(
        text_encoder, unique_items["item_content"].iloc[longest_index]
    ))
    assert sum(len(features["input_ids"]) - special_tokens for features, _ in probe_chunks) == int(kept_lengths[longest_index])
    assert all(len(features["input_ids"]) <= TEXT_CHUNK_SIZE for features, _ in probe_chunks)

    vectors = embed_text(
        text_encoder, unique_items["item_content"].tolist(),
        batch_size=TEXT_BATCH_SIZE, show_progress=True
    )
    assert vectors.shape == (len(unique_items), text_encoder.get_sentence_embedding_dimension())
    assert np.allclose(np.linalg.norm(vectors, axis=1), 1.0, atol=1e-5)
    return vectors

vectors = cached_prompt_embeddings(
    prompts,
    encoder_dir=TEXT_ENCODER_DIR,
    cache_dir=WORKSPACE / "prompt-embedding-cache",
    settings={
        "max_length": TEXT_MAX_LENGTH, "chunk_size": TEXT_CHUNK_SIZE,
        "pooling": "token_weighted_mean_l2", "dtype": "float32",
    },
    encoding_functions=(make_text_encoder, prepare_chunk_features, prompt_chunks, embed_text),
    compute=compute_prompt_vectors,
)
question_vectors = dict(zip(prompts, vectors))
del vectors


In [ ]:
# Training settings
RUN_NAME = "approach1_keras_bce_v2_dropout"
RUN_DIR = WORKSPACE / "runs" / RUN_NAME
ZIP_PATH = WORKSPACE / "dist" / f"{RUN_NAME}.zip"
SEED = 5814
MAX_ITEM_GROUPS = None
VALIDATION_FRACTION = 0.20
HIDDEN_UNITS = 64
DROPOUT_RATE = 0.2
L1_REG = 0.0
L2_REG = 1e-4
EPOCHS = 50
BATCH_SIZE = 128
LEARNING_RATE = 0.001
keras.utils.set_random_seed(SEED)
if DEVICE.type == "mps":
    torch.mps.manual_seed(SEED)
assert "question_vectors" in globals(), "Run the embeddings cell first."


In [ ]:
# Model architecture: edit layers here; tune dropout and L1/L2 in Training settings.
# Purpose: Build the classifier: a ReLU hidden layer, dropout, and one output logit, with L1/L2 weight penalties.
def make_classifier(input_dim):
    regularizer = keras.regularizers.L1L2(l1=L1_REG, l2=L2_REG)
    return keras.Sequential([
        keras.Input(shape=(input_dim,), dtype="float32", name="features"),
        keras.layers.Dense(
            HIDDEN_UNITS, activation="relu", kernel_regularizer=regularizer,
            dtype="float32", name="hidden"
        ),
        keras.layers.Dropout(DROPOUT_RATE, name="dropout"),
        keras.layers.Dense(
            1, kernel_regularizer=regularizer, dtype="float32", name="logit"
        ),
    ], name="approach1_keras")


In [ ]:
# Purpose: Split item/prompt groups into training and validation sets so shared items or prompts cannot leak across.
def split_items(observations):
    observations = observations.copy()
    items = observations[["item_id", "item_content"]].drop_duplicates()
    parent = {}

    # Purpose: Find an item or prompt group's representative, shortening links for efficient duplicate grouping.
    def find(key):
        parent.setdefault(key, key)
        while parent[key] != key:
            parent[key] = parent[parent[key]]
            key = parent[key]
        return key

    for item_id, prompt in items.itertuples(index=False, name=None):
        item_key = "item:" + str(item_id)
        prompt = "" if pd.isna(prompt) else str(prompt).strip()
        if prompt:
            text_key = "text:" + hashlib.sha256(prompt.encode()).hexdigest()
            parent[find(item_key)] = find(text_key)
        else:
            find(item_key)

    observations["item_group"] = observations["item_id"].map(
        lambda item_id: find("item:" + str(item_id))
    )
    groups = np.array(sorted(observations["item_group"].unique()))
    if MAX_ITEM_GROUPS is not None and len(groups) > MAX_ITEM_GROUPS:
        groups = np.random.default_rng(SEED).choice(groups, MAX_ITEM_GROUPS, replace=False)
        observations = observations.loc[observations["item_group"].isin(groups)].copy()

    train_groups, validation_groups = train_test_split(
        sorted(groups), test_size=VALIDATION_FRACTION, random_state=SEED
    )
    train = observations.loc[observations["item_group"].isin(train_groups)].reset_index(drop=True)
    validation = observations.loc[observations["item_group"].isin(validation_groups)].reset_index(drop=True)

    assert set(train["item_id"]).isdisjoint(validation["item_id"])
    assert set(train["item_group"]).isdisjoint(validation["item_group"])
    assert len(train) + len(validation) == len(observations)
    train_keys = set(zip(train["dataset"], train["response_id"]))
    validation_keys = set(zip(validation["dataset"], validation["response_id"]))
    assert train_keys.isdisjoint(validation_keys)
    assert train_keys | validation_keys == set(zip(observations["dataset"], observations["response_id"]))
    return train, validation

train_frame, validation_frame = split_items(flat_binary)
split_counts = pd.concat([
    train_frame.assign(split="train"),
    validation_frame.assign(split="validation")
]).groupby(["dataset", "split"]).agg(
    items=("item_id", "nunique"), subjects=("subject_id", "nunique"),
    attempts=("response_id", "size")
)
display(split_counts)


In [ ]:
# Purpose: Extract only the selected subject metadata fields and format them for the metadata encoder.
def subject_records(observations):
    subjects = observations[[f"subject_{column}" for column in SUBJECT_COLUMNS]].copy()
    subjects.columns = SUBJECT_COLUMNS
    return subjects.fillna("").astype(str).to_dict("records")

metadata_encoder = MetadataEncoder.fit(subject_records(train_frame))
print("Metadata dimensions:", metadata_encoder.dimensions)
unseen_subject = {column: "__unseen__" for column in SUBJECT_COLUMNS}
assert metadata_encoder.transform([unseen_subject]).shape == (1, metadata_encoder.dimensions)


In [ ]:
# Purpose: Concatenate one-hot subject metadata and cached prompt embeddings into each model input vector.
def make_features(observations):
    metadata = metadata_encoder.transform(subject_records(observations))
    prompts = observations["item_content"].fillna("").astype(str)
    questions = np.stack([question_vectors[prompt] for prompt in prompts])
    features = np.concatenate([metadata, questions], axis=1).astype(np.float32)
    assert np.isfinite(features).all()
    return features

train_x = make_features(train_frame)
validation_x = make_features(validation_frame)
train_y = train_frame["response"].to_numpy(dtype=np.float32)
assert train_x.shape[1] == validation_x.shape[1]
print("Training:", train_x.shape, "Validation:", validation_x.shape)


In [ ]:
# Purpose: Run batched inference with dropout off and return the model's raw output logits.
def predict_logits(model, features, batch_size=2048):
    with keras.device(KERAS_DEVICE):
        return model.predict(features, batch_size=batch_size, verbose=0).reshape(-1)

# Purpose: Convert output logits into success probabilities using a numerically stable sigmoid.
def predict_probabilities(model, features, batch_size=2048):
    logits = predict_logits(model, features, batch_size).astype(np.float64)
    return np.exp(-np.logaddexp(0.0, -logits))

# Purpose: Return validation Brier with equal response-row weight and with equal subject-dataset pair weight.
def brier_scores(observations, predictions):
    errors = observations[["dataset", "subject_id"]].copy()
    errors["squared_error"] = (np.asarray(predictions).reshape(-1) - observations["response"].to_numpy()) ** 2
    return float(errors["squared_error"].mean()), float(
        errors.groupby(["dataset", "subject_id"])["squared_error"].mean().mean()
    )

# Purpose: Measure mean BCE with dropout off and fixed weights, excluding regularization penalties.
def evaluate_bce(model, features, targets, batch_size=2048):
    logits = predict_logits(model, features, batch_size).astype(np.float64)
    targets = np.asarray(targets, dtype=np.float64).reshape(-1)
    return float(np.mean(np.logaddexp(0.0, logits) - targets * logits))


In [ ]:
# Purpose: Train with Adam and BCE; record epoch metrics and restore the best equal-pair Brier weights after early stopping.
def train_model(train_features, targets, validation_features, validation_observations):
    keras.utils.set_random_seed(SEED)
    targets = np.asarray(targets, dtype=np.float32).reshape(-1, 1)
    validation_targets = validation_observations["response"].to_numpy(dtype=np.float32, copy=True).reshape(-1, 1)
    with keras.device(KERAS_DEVICE):
        model = make_classifier(train_features.shape[1])
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE, epsilon=1e-8),
            loss=keras.losses.BinaryCrossentropy(from_logits=True),
            metrics=[keras.metrics.BinaryCrossentropy(name="BCE", from_logits=True)],
            jit_compile=False
        )

        # Purpose: Add epoch-end training BCE and validation Brier scores to the training log before early stopping checks.
        def record_scores(epoch, logs):
            row_brier, pair_brier = brier_scores(
                validation_observations, predict_probabilities(model, validation_features)
            )
            logs.update(
                training_eval_BCE=evaluate_bce(model, train_features, targets),
                validation_row_Brier=row_brier, validation_equal_pair_Brier=pair_brier
            )

        fitted = model.fit(
            train_features, targets, validation_data=(validation_features, validation_targets),
            batch_size=BATCH_SIZE, epochs=EPOCHS, shuffle=True, verbose=2,
            callbacks=[
                keras.callbacks.LambdaCallback(on_epoch_end=record_scores),
                keras.callbacks.EarlyStopping(
                    monitor="validation_equal_pair_Brier", mode="min",
                    patience=5, restore_best_weights=True
                ),
            ],
        )
    results = pd.DataFrame(fitted.history).rename(columns={
        "BCE": "training_BCE", "val_BCE": "validation_BCE",
        "loss": "training_objective", "val_loss": "validation_objective",
    })
    results = results[[
        "training_BCE", "validation_BCE", "training_eval_BCE",
        "validation_row_Brier", "validation_equal_pair_Brier",
        "training_objective", "validation_objective",
    ]]
    results.insert(0, "epoch", np.arange(1, len(results) + 1))
    best_epoch = int(results.loc[results["validation_equal_pair_Brier"].idxmin(), "epoch"])
    return model, results, best_epoch

model, history, best_epoch = train_model(train_x, train_y, validation_x, validation_frame)
display(history)
model.summary()
print("Selected epoch:", best_epoch, "Device:", next(model.parameters()).device)


In [ ]:
validation_predictions = predict_probabilities(model, validation_x)
training_mean = float(train_y.mean())
rows = []
for name, predictions in [
    ("constant 0.5", np.full(len(validation_frame), 0.5)),
    ("training global mean", np.full(len(validation_frame), training_mean)),
    ("Approach 1", validation_predictions),
]:
    row_score, pair_score = brier_scores(validation_frame, predictions)
    rows.append({"predictor": name, "row_Brier": row_score, "equal_pair_Brier": pair_score})

comparison = pd.DataFrame(rows)
display(comparison)
per_pair = validation_frame[["dataset", "subject_id"]].copy()
per_pair["squared_error"] = (
    validation_predictions - validation_frame["response"].to_numpy()
) ** 2
display(
    per_pair.groupby(["dataset", "subject_id"])["squared_error"].mean()
    .groupby("dataset").mean().rename("equal_subject_Brier").to_frame()
)


In [ ]:
# Plot 1: Training and validation curves
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

PLOT_RUN_DIR = globals().get("RUN_DIR", Path.cwd() / "runs" / "approach1_keras_bce_v1")
plot_history = history if "history" in globals() else pd.read_csv(PLOT_RUN_DIR / "training_history.csv")
plot_selected_epoch = best_epoch if "best_epoch" in globals() else json.loads((PLOT_RUN_DIR / "run.json").read_text())["selected_epoch"]

# Purpose: Plot training BCE and validation Brier over epochs, marking the selected equal-pair Brier epoch.
def plot_training_progress(results, selected_epoch):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
    results.plot(x="epoch", y="training_BCE", ax=axes[0], marker="o", legend=False)
    axes[0].set(title="Training loss", ylabel="Binary cross-entropy (lower is better)")
    results.plot(x="epoch", y=["validation_equal_pair_Brier", "validation_row_Brier"], ax=axes[1], marker="o")
    axes[1].legend(["Equal subject-benchmark weight", "Equal response-row weight"], fontsize=8)
    axes[1].axvline(selected_epoch, color="tab:green", linestyle=":")
    selected = results.loc[results["epoch"].eq(selected_epoch), "validation_equal_pair_Brier"].iloc[0]
    axes[1].scatter([selected_epoch], [selected], color="tab:green", s=70, zorder=5)
    axes[1].annotate(f"Selected epoch: {selected_epoch}", (selected_epoch, selected), xytext=(8, 12), textcoords="offset points")
    axes[1].set(title="Held-out validation score", ylabel="Brier score (lower is better)")
    for axis in axes:
        axis.set_xlabel("Epoch")
        axis.xaxis.set_major_locator(MaxNLocator(integer=True))
        axis.grid(alpha=0.25)
    plt.show()
    return fig

progress_figure = plot_training_progress(plot_history, plot_selected_epoch)


In [ ]:
# Plot 2: Validation comparison with baselines
plot_comparison = comparison.copy() if "comparison" in globals() else pd.read_csv(PLOT_RUN_DIR / "validation_comparison.csv")

# Purpose: Compare the selected model with constant-probability baselines using both Brier weightings.
def plot_validation_comparison(results):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
    colors = ["tab:green" if name == "Approach 1" else "gray" for name in results["predictor"]]
    for axis, metric, title in zip(
        axes, ["equal_pair_Brier", "row_Brier"],
        ["Equal subject-benchmark weight", "Equal response-row weight"]
    ):
        bars = axis.barh(results["predictor"], results[metric], color=colors)
        axis.bar_label(bars, fmt="%.4f", padding=5)
        axis.set_xlim(0, max(0.01, float(results[metric].max())) * 1.25)
        axis.set(title=title, xlabel="Validation Brier (lower is better)")
        axis.invert_yaxis()
        axis.grid(axis="x", alpha=0.25)
        axis.set_axisbelow(True)
    fig.suptitle("Selected Approach 1 model versus constant baselines")
    plt.show()
    return fig

comparison_figure = plot_validation_comparison(plot_comparison)


In [ ]:
# Plot 3: Training versus validation loss (overfitting)
loss_curve_history = history if "history" in globals() else pd.read_csv(PLOT_RUN_DIR / "training_history.csv")

# Purpose: Compare epoch-end training and validation BCE to reveal overfitting and mark the lowest validation BCE.
def plot_loss_learning_curve(results):
    assert {"epoch", "training_eval_BCE", "validation_BCE"} <= set(results), "Rerun training to record epoch-end BCE."
    fig, ax = plt.subplots(figsize=(9, 4.5), layout="constrained")
    curves = results.rename(columns={"training_eval_BCE": "Training BCE (epoch-end)", "validation_BCE": "Validation BCE"})
    curves.plot(x="epoch", y=["Training BCE (epoch-end)", "Validation BCE"], ax=ax, marker="o")
    minimum = results.loc[results["validation_BCE"].idxmin()]
    ax.scatter([minimum["epoch"]], [minimum["validation_BCE"]], s=85, color="tab:orange", zorder=5)
    ax.axvline(minimum["epoch"], color="gray", linestyle=":", label=f"Lowest validation BCE: epoch {int(minimum['epoch'])}")
    ax.set(title="Learning curve: training versus validation loss", xlabel="Epoch", ylabel="Binary cross-entropy (lower is better)")
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.grid(alpha=0.25)
    ax.legend()
    plt.show()
    return fig

loss_learning_curve_figure = plot_loss_learning_curve(loss_curve_history)


In [ ]:
# Purpose: Save the trained classifier, encoder, prediction helpers, and run evidence, then package the submission ZIP.
def export_submission(model):
    assert not RUN_DIR.exists(), "Choose a new RUN_NAME to preserve your previous run."
    assert not ZIP_PATH.exists(), "Choose a new RUN_NAME to preserve your previous ZIP."
    RUN_DIR.mkdir(parents=True, exist_ok=False)
    bundle = RUN_DIR / "submission"
    bundle.mkdir()
    shutil.copy2(WORKSPACE / "keras_submission.py", bundle / "model.py")
    keras_code.write_feature_helpers(WORKSPACE / "feature_utils.py", bundle / "feature_utils.py")
    shutil.copytree(
        TEXT_ENCODER_DIR, bundle / "text_encoder",
        ignore=shutil.ignore_patterns(".cache", "source.json", "README.md", "data_config.json")
    )
    config = {
        "input_dim": int(train_x.shape[1]),
        "hidden_units": HIDDEN_UNITS,
        "text_max_length": TEXT_MAX_LENGTH,
        "text_chunk_size": TEXT_CHUNK_SIZE,
        "text_pooling": "token_weighted_mean_l2",
        "categories": metadata_encoder.categories,
    }
    (bundle / "config.json").write_text(json.dumps(config, indent=2))
    with keras.device("cpu"):
        cpu_model = keras.Sequential.from_config(model.get_config())
        cpu_model.set_weights(model.get_weights())
        cpu_model.save(bundle / "classifier.keras")
    (bundle / "models.txt").write_text("sentence-transformers/all-MiniLM-L6-v2\n")
    (bundle / "requirements.txt").write_text(
        f"keras=={keras.__version__}\nsentence-transformers>=3.3,<6\n"
    )
    (bundle / "prediction_config.json").write_text(json.dumps({"concurrency": 1}))

    data_manifest = json.loads((WORKSPACE / "data_manifest.json").read_text())
    encoder_source = json.loads((WORKSPACE / "encoder_source.json").read_text())
    manifest = {
        "seed": SEED, "data_revision": data_manifest["revision"],
        "text_encoder": encoder_source,
        "validation_fraction": VALIDATION_FRACTION,
        "max_item_groups": MAX_ITEM_GROUPS, "hidden_units": HIDDEN_UNITS,
        "epochs": EPOCHS, "selected_epoch": best_epoch,
        "training_loss": "bce", "training_weighting": "response_row",
        "selection_metric": "validation_equal_pair_Brier",
        "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE,
        "text_max_length": TEXT_MAX_LENGTH,
        "text_chunk_size": TEXT_CHUNK_SIZE,
        "text_pooling": "token_weighted_mean_l2",
        "training_device": str(DEVICE),
        "framework": "keras", "keras_version": keras.__version__,
        "backend": keras.backend.backend(), "adam_epsilon": 1e-8,
        "kernel_initializer": "glorot_uniform", "bias_initializer": "zeros",
        "classifier_config": model.get_config(),
        "training_objective": "bce_plus_kernel_l1_l2",
        "splits": {
            name: {
                "item_groups": sorted(part["item_group"].unique()),
                "responses": part[["dataset", "response_id"]].to_dict("records")
            }
            for name, part in [("train", train_frame), ("validation", validation_frame)]
        }
    }
    (RUN_DIR / "run.json").write_text(json.dumps(manifest, indent=2))
    history.to_csv(RUN_DIR / "training_history.csv", index=False)
    comparison.to_csv(RUN_DIR / "validation_comparison.csv", index=False)

    ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "x", compression=zipfile.ZIP_DEFLATED) as archive:
        for file in sorted(bundle.rglob("*")):
            if file.is_file():
                archive.write(file, file.relative_to(bundle).as_posix())
    return bundle, ZIP_PATH

submission_dir, submission_zip = export_submission(model)
print(submission_dir)
print(submission_zip)


In [ ]:
# Purpose: Convert a flattened validation row into the subject/item input format expected by the exported predictor.
def predictor_input(row):
    subject_fields = [
        "normalized_name", "provider", "release_date", "access_date",
        "harness", "harness_version", "reasoning_effort", "subject_features_extra"
    ]
    # Purpose: Convert metadata values to strings, replacing missing values with an empty string.
    def string_value(value):
        return "" if pd.isna(value) else str(value)

    subject = {
        field: string_value(row[f"subject_{field}"]) for field in subject_fields
    }
    item = {
        "item_content": string_value(row["item_content"]),
        "item_features": string_value(row["item_item_features"]),
        "interactors": string_value(row["interactors"]),
        "benchmark_id": "benchmark_local_check",
    }
    return [subject, item]

# Purpose: Check that exported predictions match notebook probabilities and handle unseen metadata and revealed labels.
def check_exported_predictor(bundle):
    specification = importlib.util.spec_from_file_location("exported_approach1_keras", bundle / "model.py")
    exported = importlib.util.module_from_spec(specification)
    specification.loader.exec_module(exported)
    for index, row in validation_frame.head(5).iterrows():
        example = predictor_input(row)
        probability = exported.predict(example, [])
        assert isinstance(probability, float) and np.isfinite(probability)
        assert 0 <= probability <= 1
        assert np.isclose(probability, validation_predictions[index], atol=1e-4, rtol=1e-4)
        assert probability == exported.predict(example, [[example, 1]])
    unseen = predictor_input(validation_frame.iloc[0])
    unseen[0]["normalized_name"] = "__unseen_subject__"
    assert 0 <= exported.predict(unseen, []) <= 1
    return True

assert check_exported_predictor(submission_dir)
print("Exported predictor checks passed.")


In [ ]:
# Purpose: Run the official submission ZIP validator and predictor smoke test.
def run_official_checks(bundle, archive):
    subprocess.run(
        [sys.executable, str(KIT_DIR / "tools/check_submission_zip.py"), str(archive)],
        cwd=KIT_DIR, check=True
    )
    subprocess.run(
        [sys.executable, str(KIT_DIR / "tools/run_smoke_test.py"),
         str(bundle), "--concurrency", "1"],
        cwd=KIT_DIR, check=True
    )

assert (KIT_DIR / "tools/check_submission_zip.py").is_file(), "Set AIMS_STARTER_KIT_DIR to the official starting kit before running this optional check."
run_official_checks(submission_dir, submission_zip)
